# DUR Risk Map — Key Findings

This notebook documents four analytical findings derived from Korea's public DUR (Drug Utilization Review) contraindication data.  
The network has **386 nodes** (drugs) and **844 edges** (contraindication pairs) sourced from the AC (absolute contraindication) file.

---

In [ ]:
import pandas as pd
import networkx as nx
from networkx.algorithms import community
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

base = Path('../data/processed')
edge = pd.read_csv(base / 'ac_edge_table_english.csv')
node = pd.read_csv(base / 'ac_node_table_with_overlay.csv')

G = nx.Graph()
for _, r in node.iterrows():
    G.add_node(r['code'], **r.to_dict())
for _, r in edge.iterrows():
    G.add_edge(r['source_code'], r['target_code'], raw_count=r['raw_count'])

print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')

---
## Finding 1 — Most Documented Contraindication Pairs

The `raw_count` field counts how many raw data rows support each contraindication pair.  
A high count indicates the pair is documented across many product formulations, suggesting a **well-established clinical interaction**.

Two dominant patterns emerge:  
- **Iodinated contrast agents + Metformin** → risk of lactic acidosis via contrast-induced acute kidney injury (AKI)
- **Rosuvastatin + CYP3A4/OATP inhibitors** → risk of rhabdomyolysis via impaired statin clearance

In [ ]:
top10 = (
    edge[['source_label_eng', 'target_label_eng', 'reason', 'raw_count']]
    .sort_values('raw_count', ascending=False)
    .drop_duplicates(subset=['source_label_eng', 'target_label_eng'])
    .head(10)
    .reset_index(drop=True)
)
top10.index += 1
print(top10.to_string())

In [ ]:
fig = px.bar(
    top10.assign(pair=top10['source_label_eng'] + ' ↔ ' + top10['target_label_eng']),
    x='raw_count', y='pair', orientation='h',
    color='reason',
    title='Top 10 Drug Pairs by Raw Count (Evidence Strength)',
    labels={'raw_count': 'Raw Count', 'pair': 'Drug Pair', 'reason': 'Reason'},
    height=420
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=True)
fig.show()

**Key takeaway:** 7 of the top 10 pairs involve Metformin + an iodinated contrast agent, all sharing the lactic acidosis mechanism. This reflects the large number of contrast formulations registered in Korea rather than independent clinical evidence. Rosuvastatin pairs (rhabdomyolysis) are the only truly distinct cluster in the top-10.

---
## Finding 2 — Hub Drugs vs Bridge Drugs

**Degree centrality** captures how many contraindication partners a drug has (breadth of risk).  
**Betweenness centrality** captures how often a drug sits on the shortest path between other drugs (structural bottleneck).

Drugs that score high on both are the most clinically dangerous: they're involved in many interactions *and* their removal fragments the network.  
Drugs that score high only on betweenness are "bridge" drugs — modest partner count but disproportionate structural influence.

In [ ]:
degree = pd.Series(dict(G.degree()), name='degree')
betweenness = pd.Series(nx.betweenness_centrality(G), name='betweenness')
centrality = pd.concat([degree, betweenness], axis=1)
centrality['label_eng'] = [G.nodes[n].get('label_eng', n) for n in centrality.index]

# Thresholds: top 25% for hub, top 10% for bridge
deg_thresh   = centrality['degree'].quantile(0.75)
bet_thresh   = centrality['betweenness'].quantile(0.90)

def classify(row):
    h = row['degree'] >= deg_thresh
    b = row['betweenness'] >= bet_thresh
    if h and b: return 'Hub + Bridge'
    if h:        return 'Hub Only'
    if b:        return 'Bridge Only'
    return 'Peripheral'

centrality['role'] = centrality.apply(classify, axis=1)

print('--- Role distribution ---')
print(centrality['role'].value_counts())
print('\n--- Top Bridge-Only drugs (high influence, few direct connections) ---')
print(
    centrality[centrality['role']=='Bridge Only']
    .sort_values('betweenness', ascending=False)[['label_eng','degree','betweenness']]
    .head(8).to_string(index=False)
)
print('\n--- Top Hub+Bridge drugs (highest overall risk) ---')
print(
    centrality[centrality['role']=='Hub + Bridge']
    .sort_values('betweenness', ascending=False)[['label_eng','degree','betweenness']]
    .head(8).to_string(index=False)
)

In [ ]:
color_map = {
    'Hub + Bridge': '#d62728',
    'Hub Only':     '#ff7f0e',
    'Bridge Only':  '#1f77b4',
    'Peripheral':   '#aec7e8'
}
fig2 = px.scatter(
    centrality.reset_index(),
    x='degree', y='betweenness',
    color='role',
    color_discrete_map=color_map,
    hover_name='label_eng',
    title='Degree vs Betweenness Centrality',
    labels={'degree': 'Degree (# contraindication partners)', 'betweenness': 'Betweenness Centrality'},
    opacity=0.75, height=460
)
fig2.show()

**Key takeaway:**  
- **Tizanidine** is the most extreme bridge drug (degree=2 but betweenness=0.175) — it connects two otherwise separate sub-clusters.  
- **Fluvoxamine** and **Ciprofloxacin** are the most impactful mid-size connectors (degree 3-4, betweenness ~0.17).  
- **SelegilineHydrochloride**, **Nirmatrelvir**, and **Pimozide** are the top Hub+Bridge drugs — highest combined risk profile.

---
## Finding 3 — Multi-Vulnerable-Population Drugs

Each drug node is annotated with three population flags:
- `is_preg_contra` — contraindicated during pregnancy (AC type)
- `is_elderly_caution` — requires caution in elderly patients
- `is_age_contra` — contraindicated in specific age groups (pediatric/geriatric)

Drugs that carry **all three flags** present the broadest prescribing risk:  
they must be avoided or managed carefully across the most vulnerable populations simultaneously.

In [ ]:
flag_cols = ['is_preg_contra', 'is_elderly_caution', 'is_age_contra']
for col in flag_cols:
    node[col] = node[col].astype(bool)

print('--- Flag counts (of 386 total drugs) ---')
for col in flag_cols:
    print(f'  {col:25s}: {node[col].sum():3d}')

triple = node[node[flag_cols].all(axis=1)][['label_eng','label_kor','degree']].sort_values('degree', ascending=False)
dual   = node[node['is_preg_contra'] & node['is_elderly_caution']][['label_eng','label_kor','degree']].sort_values('degree', ascending=False)

print(f'\n=== Triple-flag drugs (all 3 populations): {len(triple)} drugs ===')
print(triple.to_string(index=False))

print(f'\n=== Pregnancy + Elderly dual-flag: {len(dual)} drugs (top 10) ===')
print(dual.head(10).to_string(index=False))

In [ ]:
from matplotlib_venn import venn3
# Fallback: bar chart showing flag combination counts
node['flag_count'] = node[flag_cols].sum(axis=1)
fig3 = px.histogram(
    node, x='flag_count', nbins=4,
    title='Distribution of Vulnerable-Population Flag Count per Drug',
    labels={'flag_count': 'Number of Vulnerable-Population Flags', 'count': '# Drugs'},
    color_discrete_sequence=['#636EFA']
)
fig3.update_xaxes(tickmode='linear', dtick=1)
fig3.show()

**Key takeaway:**  
- 304/386 drugs (78.8%) are already pregnancy-contraindicated, making pregnancy the dominant flag by far.  
- Only **11 drugs** carry all three flags. They are dominated by **psychotropic agents** (antipsychotics: Chlorpromazine, Amisulpride, Perphenazine, Chlorprothixene, Thiothixene; tricyclic antidepressant: Nortriptyline) and **NSAIDs** (Ketoprofen, Meloxicam, Dexketoprofen, Lornoxicam) — drug classes that are routinely prescribed and therefore especially important to flag.  
- The elderly caution flag is the rarest (43 drugs = 11.1%), suggesting this data source undercaptures geriatric risk.

---
## Finding 4 — Network Communities

Louvain community detection partitions the network into groups of drugs that share more interactions with each other than with the rest of the network.  
These communities often correspond to shared **pharmacological mechanisms** (same enzyme target, receptor class, or metabolic pathway).

In [ ]:
comms = community.louvain_communities(G, seed=42)
comms_sorted = sorted(comms, key=len, reverse=True)

rows = []
manual_labels = [
    'CYP450 Inducers (Rifampicin)',
    'Antiarrhythmics / QT-prolonging',
    'CYP3A4 Inhibitors (Antifungals)',
    'MAOIs & Antipsychotics (Selegiline)',
    'NSAIDs & Pain-related',
    'Immunosuppressants (Calcineurin)',
    'HIV Antiretrovirals',
    'PDE5 Inhibitors + Nitrates',
    'Iodinated Contrast + Metformin',
    'Retinoids',
]
for i, c in enumerate(comms_sorted[:10]):
    members = list(c)
    top3 = sorted(members, key=lambda n: G.degree(n), reverse=True)[:3]
    labels = [G.nodes[n].get('label_eng', n) for n in top3]
    rows.append({
        'Rank': i+1,
        'Size': len(c),
        'Avg Degree': round(sum(G.degree(n) for n in members)/len(members), 1),
        'Preg Flags': sum(1 for n in members if G.nodes[n].get('is_preg_contra', False)),
        'Elder Flags': sum(1 for n in members if G.nodes[n].get('is_elderly_caution', False)),
        'Top-3 Hubs': ', '.join(labels),
        'Mechanism Theme': manual_labels[i] if i < len(manual_labels) else ''
    })

df_comms = pd.DataFrame(rows).set_index('Rank')
print(df_comms.to_string())

In [ ]:
fig4 = px.bar(
    df_comms.reset_index(),
    x='Mechanism Theme', y='Size',
    color='Elder Flags',
    hover_data=['Avg Degree', 'Preg Flags', 'Top-3 Hubs'],
    title='Top 10 Network Communities — Size and Elderly Caution Burden',
    labels={'Size': 'Number of Drugs', 'Elder Flags': 'Elderly Caution Flags'},
    color_continuous_scale='Oranges',
    height=450
)
fig4.update_xaxes(tickangle=30)
fig4.show()

**Key takeaway:**  
- The algorithm found **31 communities** total; the top 10 cover 328/386 drugs (85%).  
- Community structure closely mirrors known pharmacology: CYP450 substrate/inhibitor/inducer groupings, receptor-class clusters (antipsychotics, PDE5), and shared metabolic interaction points (calcineurin inhibitors, retinoids).  
- **Community 5 (NSAIDs)** stands out with the highest elderly caution burden (22 drugs, 58% of community) — NSAIDs are among the most widely prescribed drugs in elderly patients despite known renal/GI risk.  
- **Community 9 (Contrast + Metformin)** is the smallest dense cluster but carries the most homogeneous mechanism (all edges share lactic acidosis reason).

---

## Summary

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Contrast agents + Metformin dominate by raw_count | Well-documented lactic acidosis risk; high formulation count inflates raw_count |
| 2 | Tizanidine, Fluvoxamine bridge separate clusters | Removing these drugs would fragment the network; high prescribing caution needed |
| 3 | 11 drugs flagged for all 3 vulnerable populations | Mostly antipsychotics and NSAIDs — common prescribing agents with broad restriction |
| 4 | 31 Louvain communities mirror known pharmacology | Network structure is mechanistically coherent; useful for curriculum or guideline design |
| 5 | 38 pairs are both GC-overlapping AND AC-contraindicated | Ketorolac vs any NSAID and triptan vs triptan carry a compounded regulatory signal from two independent rule sources |
| 6 | 6 of top-20 hubs are also DC dose-caution drugs | Ketorolac appears in both F5 and F6 — most multiply-constrained drug across all rule types in this dataset |

---
## Finding 5 — Double-Hit Pairs: Pharmacological Overlap AND Absolute Contraindication

The GC (효능군중복) file flags drugs that belong to the same pharmacological class — co-prescribing two drugs from the same GC class is considered a "functional duplication" risk.

Cross-referencing GC class membership with AC contraindication edges reveals **double-hit pairs**: drug combinations that are simultaneously:
1. **Functionally duplicant** (same GC class → additive/redundant effect)
2. **Absolutely contraindicated** (AC edge → serious adverse event on record)

These pairs carry a compounded risk signal not visible from either dataset alone.

In [ ]:
gc = pd.read_csv('../data/raw/OpenData_PotOpenDurIngr_GC20260312.csv', encoding='utf-8-sig', low_memory=False)

class_to_codes = gc.groupby('계열명')['DUR성분코드'].apply(set).to_dict()

double_hit_pairs = []
for cls, codes in class_to_codes.items():
    codes = [str(c) for c in codes]
    for i, c1 in enumerate(codes):
        for c2 in codes[i+1:]:
            match = edge[
                ((edge['source_code'].astype(str) == c1) & (edge['target_code'].astype(str) == c2)) |
                ((edge['source_code'].astype(str) == c2) & (edge['target_code'].astype(str) == c1))
            ]
            if len(match) > 0:
                row = match.iloc[0]
                double_hit_pairs.append({
                    'GC Class': cls,
                    'Drug A': row.get('source_label_eng', c1),
                    'Drug B': row.get('target_label_eng', c2),
                    'AC Reason': row.get('reason', '-'),
                    'Raw Count': row.get('raw_count', 0)
                })

df_dh = pd.DataFrame(double_hit_pairs)
print(f'Total double-hit pairs: {len(df_dh)}')
print(f'Classes involved: {df_dh["GC Class"].nunique()}')
print()
print(df_dh['GC Class'].value_counts())

In [ ]:
fig5a = px.bar(
    df_dh['GC Class'].value_counts().reset_index().rename(columns={'index': 'GC Class', 'count': 'Double-Hit Pairs'}),
    x='GC Class', y='Double-Hit Pairs',
    title='Double-Hit Pairs by Pharmacological Class (GC Overlap + AC Contraindication)',
    labels={'GC Class': 'GC Pharmacological Class'},
    color='Double-Hit Pairs',
    color_continuous_scale='Reds',
    height=400
)
fig5a.update_xaxes(tickangle=30)
fig5a.show()

# Detailed table for NSAIDs (dominant cluster)
nsaid_pairs = df_dh[df_dh['GC Class'] == '비스테로이드성소염제'].copy()
nsaid_pairs = nsaid_pairs.sort_values('Drug A')
print(f"\nNSAID double-hit pairs ({len(nsaid_pairs)} total):")
print(nsaid_pairs[['Drug A', 'Drug B', 'AC Reason']].to_string(index=False))

**Key takeaway:**  
- **38 double-hit pairs** were found across 3 GC classes: NSAIDs (31), Triptans/선택적5-HT1수용체효능제 (6), and Ergot alkaloids/맥각알칼로이드계편두통치료제 (1).  
- **31 NSAID pairs carry dual regulatory signals**: all members of the GC NSAID class already face "functional duplication" risk (GC flag), but 31 specific pairings are *also* documented as absolute contraindications (AC edges), mostly for severe GI adverse events. This compounding signal is strongest.  
- **All triptans (Sumatriptan, Zolmitriptan, Naratriptan, Almotriptan, Frovatriptan) form a near-complete AC clique within their GC class** — any two triptans together risk additive vasoconstrictive crisis.  
- These pairs are the highest clinical priority in the dataset: two independent regulatory systems (GC + AC) are both flagging the same combinations, providing a compounded safety signal.

---
## Finding 6 — Hub Drugs Under Dose Caution (DC)

The DC (용량주의) file flags drugs requiring dose adjustment due to known risk at standard doses.  
Combining this with the contraindication network reveals drugs under **dual pressure**: they are both widely contraindicated with other drugs (high degree in the AC graph) AND require dose management when used at all.

These drugs represent the highest-priority prescribing complexity: more contraindication partners means fewer options, while dose caution means the drug itself is harder to use safely.

In [ ]:
dc = pd.read_csv('../data/raw/OpenData_PotOpenDurIngr_DC20260312.csv', encoding='utf-8-sig', low_memory=False)
dc_codes = set(dc['DUR성분코드'].dropna().astype(str))

node['is_dose_caution'] = node['code'].astype(str).isin(dc_codes)

print(f'DC-flagged drugs: {node["is_dose_caution"].sum()} of {len(node)} ({node["is_dose_caution"].mean()*100:.1f}%)')
print(f'DC-flagged drugs in top-20 hubs: {node.nlargest(20, "degree")["is_dose_caution"].sum()} of 20')
print()

# Quadrant analysis: high degree + dose caution vs not
deg_median = node['degree'].median()
node['quadrant'] = node.apply(
    lambda r: 'High Degree + Dose Caution' if r['degree'] > deg_median and r['is_dose_caution']
    else ('High Degree, No Dose Caution' if r['degree'] > deg_median
    else ('Low Degree + Dose Caution' if r['is_dose_caution'] else 'Low Degree, No Caution')),
    axis=1
)
print(node['quadrant'].value_counts())

In [ ]:
color_map6 = {
    'High Degree + Dose Caution': '#d62728',
    'High Degree, No Dose Caution': '#ff7f0e',
    'Low Degree + Dose Caution': '#1f77b4',
    'Low Degree, No Caution': '#aec7e8'
}

fig6 = px.scatter(
    node,
    x='degree', y='betweenness' if 'betweenness' in node.columns else 'degree',
    color='quadrant',
    color_discrete_map=color_map6,
    hover_name='label_eng',
    hover_data={'degree': True, 'is_dose_caution': True},
    title='Hub Drugs vs Dose Caution Flag — Degree Distribution',
    labels={'degree': 'Degree (# contraindication partners)', 'quadrant': 'Risk Profile'},
    opacity=0.75,
    height=480
)
fig6.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig6.show()

# Top flagged table
top_dc_hubs = (
    node[node['is_dose_caution']]
    .nlargest(15, 'degree')[['label_eng', 'degree', 'is_preg_contra', 'is_elderly_caution', 'is_age_contra']]
    .reset_index(drop=True)
)
top_dc_hubs.index += 1
print("\nTop 15 dose-caution drugs by contraindication degree:")
print(top_dc_hubs.to_string())

**Key takeaway:**  
- **90 of 386 drugs (23.3%)** in the AC graph are also DC-flagged for dose caution.  
- Among the **top 20 hubs by degree**, 6 are also dose-caution: **SelegilineHydrochloride** (degree=40), **Ketorolac** (29), **Pimozide** (26), **Haloperidol** (24), **Amiodarone** (21), **Erythromycin** (16).  
- **Rifampicin** (degree=45, the top hub overall) is *not* DC-flagged in this dataset, while Ketorolac, Selegiline, and other top hubs are — suggesting their risk profile includes both DDI *and* dose-dependent safety concerns.  
- **Ketorolac** appears in both Finding 5 (double-hit NSAID pairs) and Finding 6 (high-degree dose-caution hub): it is the single most multiply-constrained drug in this dataset across multiple rule types.  
- This quadrant view provides a practical prescription triage framework: "High Degree + Dose Caution" drugs (red) require both a partner-avoidance check and a dosing review simultaneously.